In [ ]:
# 1. 필요한 패키지를 고정 버전으로 설치합니다. Python 3.11 또는 3.12를 사용하세요.
# Kaggle은 Internet을 켜고, Colab에서는 연결 후 '모두 실행'을 누릅니다.
# Linux에서 macOS용 libomp, MPS, 한글 폰트를 설치하지 않습니다.
import os
import subprocess
import sys
assert (3, 11) <= sys.version_info[:2] <= (3, 12), sys.version
PINS = [
    "numpy==2.3.5" if sys.version_info[:2] == (3, 11) else "numpy==2.5.4",
    "pandas==3.1.1", "scipy==1.17.1", "scikit-learn==1.6.1", "lightgbm==4.6.0",
    "torch==2.9.1", "xgboost==3.1.2" if sys.version_info[:2] == (3, 11) else "xgboost==3.4.1", "tabicl==2.2.0", "einops==0.8.2",
    "huggingface-hub==0.36.0", "psutil==7.2.2", "tqdm==4.70.1",
]
# SINCON_PIP_NO_INDEX=1은 로컬의 이미 설치한 동일 버전을 재확인할 때만 사용합니다.
pip_args = ["--no-index"] if os.environ.get("SINCON_PIP_NO_INDEX") == "1" else []
subprocess.check_call([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *pip_args, *PINS])
# 설치 전 이미 import된 이전 버전은 pip가 메모리에서 바꾸지 못합니다.
aliases = {"scikit-learn": "sklearn", "huggingface-hub": "huggingface_hub"}
stale = []
for pin in PINS:
    distribution, expected = pin.split("==")
    module = sys.modules.get(aliases.get(distribution, distribution))
    actual = str(getattr(module, "__version__", expected)).split("+")[0]
    if actual != expected:
        stale.append(f"{distribution}: loaded {actual}, required {expected}")
if stale:
    raise RuntimeError("이전 버전이 이미 로드되어 있습니다. 런타임을 다시 시작한 뒤 모두 실행하세요. " + "; ".join(stale))
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
print("설치 완료. Python:", sys.version)


# SINCON 2026 핫식스 최종 제출 재현

이 노트북은 인천 지하철 시간대별 승차 인원을 예측하고, `example_id,boardings` 두 열의 CSV를 만듭니다. 지표 RMSLE는 실제 인원과 예측 인원에 `log1p`를 씌운 뒤 차이의 제곱평균제곱근을 구합니다. 값이 작을수록 좋습니다.

현재 기본 결과는 팀의 `submission_clean_refined.csv`입니다. Public 0.148046은 사용자가 이미 완성된 파일을 제출하여 확인한 점수입니다. 다른 완성 후보 `submission_clean_B_station_k05.csv`는 0.147820입니다. 이번 작업은 어느 쪽의 예측값도 이 점수에 맞춰 바꾸지 않으며, Kaggle에 업로드하지 않습니다.

## 처음 실행하는 방법

1. Kaggle에서는 `final_package/` 전체를 **비공개 Dataset**으로 연결합니다. 폴더 이름이나 Dataset slug는 자동 탐지합니다. 노트북은 `/kaggle/working`에 결과를 씁니다.
2. Colab에서는 패키지 ZIP을 `/content` 아래에 풉니다. 노트북과 `payload_manifest.json`의 상대 구조를 유지합니다. 예: `!unzip /content/SINCON_final_package.zip -d /content`.
3. 로컬에서는 아래 `LOCAL_ROOT` 한 줄에 패키지 폴더만 지정하면 됩니다. 기본 제공 공식 CSV를 사용하므로 별도의 팀 폴더는 필요 없습니다.
4. `RUN_MODE="assemble"`로 모두 실행하면 보관된 모델 출력에서 보정을 다시 계산합니다. 기본 모델 학습은 생략하며 몇 분 내 완료를 목표로 합니다. 설치 시간은 네트워크와 기존 패키지에 따라 달라집니다.
5. `RUN_MODE="full"`로 바꾸면 아래 켜진 모든 모델 단계를 공식 train부터 다시 학습합니다. 한 단계라도 끄면 그 단계는 캐시를 사용하며, 결과 기록에도 '전체 재학습 아님'을 표시합니다.

설치 전에 다른 셀에서 이전 버전을 이미 import한 경우, 첫 셀이 이를 알리고 중단합니다. 그때는 런타임을 다시 시작한 뒤 위에서부터 실행합니다. 첫 셀을 통과하면 설치 버전과 메모리에 남은 이전 버전이 섞이지 않습니다.

규칙 4절의 7월 이후 지하철 승하차 실측은 어떤 경로로도 사용하지 않습니다. 실행 시작 시 공식 train의 마지막 날짜가 2026-06-30인지 확인하며, 검증 정답도 이 train에서 다시 조회합니다. 외부 날씨·버스·비지하철 이동량·행사 사실에는 7월 이후 정보가 들어 있습니다. 그 범위를 아래 표에 구분했습니다.


In [ ]:
from pathlib import Path
import json

LOCAL_ROOT = os.environ.get("SINCON_LOCAL_ROOT") or None  # 로컬 경로를 바꿀 때는 이 변수 하나만 수정
RUN_MODE = os.environ.get("SINCON_RUN_MODE", "assemble")
DEVICE = os.environ.get("SINCON_DEVICE", "auto")  # auto: CUDA가 있으면 사용, 없으면 CPU. cpu: CPU 고정
THREADS = int(os.environ.get("SINCON_THREADS", "4"))

FULL_STAGES = dict(
    native=True,            # 원본 LightGBM + MLP 30시드, A/B/C와 최종
    stacks=True,            # 생활이동·그래프·시간대·버스·혼합·경기·축제·주거
    rolling=True,           # 이른 기준일 6개 + 엄격 여름 MLP 30시드
    hourly_xgb=True,        # 시간별 XGBoost, A/B/C·추가 기준일·최종
    clean_oof=True,         # 새 예측에서 clean OOF를 다시 구성
    event_responses=True,   # 취소·장소 LGB 3시드와 MLP 0/1/2시드
    tabicl=True,            # 동결 체크포인트, 최종과 과거 9개 구간
    vector=True,            # 18시간 벡터 XGBoost, 과거와 최종
)
# 배치 검증 시에만 일부 단계의 이름을 쉼표로 넘깁니다. 노트북 기본값은 모두 True입니다.
if os.environ.get("SINCON_FULL_STAGES"):
    requested = set(os.environ["SINCON_FULL_STAGES"].split(","))
    assert requested <= FULL_STAGES.keys(), requested
    FULL_STAGES = {name: name in requested for name in FULL_STAGES}
if RUN_MODE == "full" and FULL_STAGES["native"] and not FULL_STAGES["event_responses"]:
    raise ValueError("native와 event_responses는 함께 켜세요. 행사 응답은 원본 모델 예측과 맞아야 합니다.")

# Claude가 공식 학습 기간 검증을 마친 추가 보정만 나중에 활성화합니다.
# enabled 한 줄을 True로 바꾸고 path/evidence의 파일을 패키지에 추가하세요.
# 현재 두 항목은 꺼져 있으므로 기준 정리본과 같은 결과를 만듭니다.
CORRECTIONS = [
    dict(name="검증된 추가 역별 보정", enabled=False, path="reviewed/station_offsets.csv", evidence="reviewed/station_evidence.json"),
    dict(name="검증된 추가 행사 보정", enabled=False, path="reviewed/event_offsets.csv", evidence="reviewed/event_evidence.json"),
]
print("모드:", RUN_MODE, "장치:", DEVICE, "스레드:", THREADS)


## 전체 흐름과 입력·출력

1. **공식 CSV 및 캐시 해시 검사**. 688,176행 train, 157,194행 test, 71역 메타데이터를 확인합니다. 외부 입력은 동결 파일을 사용합니다. 네트워크로 새 실측을 가져오는 코드는 없습니다.
2. **모델별 기본 예측**. LightGBM과 역·시간·요일 임베딩 MLP가 `log1p(승차)`를 예측합니다. MLP는 최종 및 A/B/C에서 시드 0~29를 평균합니다. 원본 앙상블의 Public 수준·역 패턴 보정은 적용하지 않습니다.
3. **조건별 결합**. 비지하철 생활이동 LightGBM, 120 epoch 그래프 신경망, 가법 시간대 모형, 버스 LightGBM, 6성분·125 epoch 수요 혼합 신경망, 야구·축구 모형을 결합합니다. 그래프는 하루 총량보다 시간대 모양을 보완하고, 버스는 주변 이동 변화의 대리변수입니다. `conditional_stack`을 만듭니다. 축제의 과거 시간대 효과와 0.5배 주거 변화층을 더해 `local_stack`을 만듭니다.
4. **초기 검증 기반 보정**. 행사 일정·접근역을 수정하고, 시간별 XGBoost 10% 혼합 및 심야 날씨, 계양·귤현·검단권·심야 프로필을 적용합니다. 과거 기준일 예측 오차로 크기를 정합니다. 출력은 `early.csv`입니다.
5. **모델층 전 행사 보정**. 취소·잘못된 장소 플래그를 제거한 동일 모델의 응답, 과거 비교 행사, 과학대제전, 원미 체육대회 등을 적용합니다. 출력은 `events.csv`와 이후 일반일 보정에서 유지할 `protected.csv`입니다.
6. **일반일 후속 모델층**. 역별 과거 프로필, TabICL 일별 이탈, 동춘 휴무, 음력 연휴, 검단 아침 피크, 18시간 벡터 XGBoost를 순서대로 적용합니다. TabICL 가중치는 각 기준일 이전 예측 오차에서 구하며, 벡터 보정은 하루 및 반복 역·요일·시간 평균을 보존합니다. 행사 보호 행은 유지합니다. 출력은 `model.csv`와 행별 검증 결과입니다.
7. **최종 행사 보정**. 검암 드림파크 43행, 과학대제전 시간대 12행, 선학 2행, 둘레길 2행을 수정합니다. 그다음 `CORRECTIONS`의 켜진 추가 목록만 적용합니다. 기본 추가 목록은 모두 꺼져 있습니다.
8. **제출 형식과 기준 차이 검사**. 공식 test의 ID 순서, 157,194행, 열 이름, 유한한 비음수 인원을 검사합니다. 기준 파일과의 최대 `log1p` 절대 차이 및 SHA-256을 출력합니다. 기준 CSV는 이 마지막 비교에만 읽습니다. 예측 함수의 입력으로 사용하지 않습니다.

assemble은 1단계 이후 2~3단계와 독립 모델의 보관 예측을 사용하고, 4~8단계의 보정·검증은 실제로 다시 계산합니다. full은 같은 위치의 캐시를 새 학습 결과로 대체합니다. 따라서 재학습 결과가 실제 최종 CSV까지 연결됩니다. PyTorch·XGBoost 버전과 하드웨어가 달라지면 full 결과는 기준과 달라질 수 있습니다.

여기서 OOF는 학습에 쓰지 않은 뒤 기간을 예측해 보관한 과거 검증 결과입니다. 그 시점에 알 수 없던 뒤 날짜의 정답을 가중치 추정에 넣지 않습니다. 예를 들어 2026-02-28 기준일의 보정은 그날까지 확보한 이전 예측 오차만 사용합니다. epoch는 신경망이 학습 자료 전체를 한 번 읽는 단위입니다.

과거 예측의 역할도 나눕니다. `earlier/`에는 심야 보정 학습에 사용한2025년 기준일4개만 둡니다. `rolling/`의2026년3·4월 보조 예측은 추가 검증에 사용합니다. 모든 학습을 끈 full과 assemble이 같은 CSV를 만드는 회귀 검사로 이 범위를 확인했습니다.

다른 구조의 모델을 결합하는 이유는 같은 날·역에서도 틀리는 시간대가 다르기 때문입니다. XGBoost는 독립적인 비선형 시간대 패턴을, TabICL은 독립적인 역·일 평균 수요 변화를 보완합니다. 마지막 행사층은 잘못된 개최 플래그나 접근역을 바로잡고, 과거 비교 행사의 시간대별 증가량을 전이합니다. 새로운 미래 지하철 정답에 맞추는 단계가 아닙니다.


In [ ]:
# 모듈을 import하기 전에 패키지의 위치를 찾습니다.
if LOCAL_ROOT:
    package = Path(LOCAL_ROOT).expanduser().resolve()
else:
    candidates = [p for base in (Path.cwd(), Path("/kaggle/input"), Path("/content"))
                  for p in (base, base / "final_package") if (p / "payload_manifest.json").is_file()]
    if not candidates:
        candidates = [p.parent for base in (Path("/kaggle/input"), Path("/content")) if base.exists()
                      for p in base.rglob("payload_manifest.json")]
    if not candidates:
        raise FileNotFoundError("final_package를 풀고 LOCAL_ROOT를 지정하세요.")
    package = candidates[0].resolve()
sys.path.insert(0, str(package))
from pipeline import Run
run = Run(package, mode=RUN_MODE, device=DEVICE, threads=THREADS)
print("패키지:", package)
print("출력 폴더:", run.output)


## 검증 구간과 확인한 효과

각 기준일까지만 모델을 학습하고 그 뒤 날짜를 예측합니다. A는 2026-02-28 → 3~6월, B는 2025-12-31 → 2026년 1~4월, C는 2025-10-31 → 2025년 11월~2026년 2월입니다. A와 B에는 같은 3~4월이 들어 있으므로 세 점수를 독립 표본처럼 합산하지 않습니다. A/C 통합은 날짜가 겹치지 않습니다.

summer는 2025-06-30 기준으로 2025년7~10월을 예측합니다. 추가 월말 기준일 2025-07-31·08-31·09-30은 각각 그 뒤4개월을 예측합니다. `disjoint_warm`은 세 예측 중8월·9월·10월만 하나씩 가져와 같은 날짜가 겹치지 않게 합친 비교입니다. `summer_established_68`은 검단 연장3역을 제외한 기존68역의 여름 비교입니다.

아래 수치는 이번 패키지 이전에 깨끗한 OOF로 측정한 기록입니다. 실행 로그의 새 점수와 구분합니다. 기준과 후보 모두 Public에서 정한 수준·역별 배율을 사용하지 않습니다. '후속 결합'은 일반일 모델층까지의 효과이며, 맨 마지막 별도 행사 59행을 포함한 전체 품질 평가가 아닙니다.


| 구간 | 행 수 | 후속 결합 전 RMSLE | 후속 결합 후 RMSLE |
|---|---:|---:|---:|
| 2025-07-31 | 155,916 | 0.181522667 | 0.179802034 |
| 2025-08-31 | 155,916 | 0.177946220 | 0.177022966 |
| 2025-09-30 | 157,194 | 0.172507694 | 0.171524834 |
| A | 155,916 | 0.150806837 | 0.149904003 |
| B | 153,360 | 0.165727453 | 0.164576151 |
| C | 153,360 | 0.162339372 | 0.161174942 |
| summer | 157,194 | 0.254875330 | 0.253163600 |
| disjoint_A_C | 309,276 | 0.156631618 | 0.155594980 |
| disjoint_warm | 117,576 | 0.175753248 | 0.174301745 |
| summer_established_68 | 150,552 | 0.189900100 | 0.187494152 |


| 단계 | A 전 → 후 | B 전 → 후 | C 전 → 후 |
|---|---:|---:|---:|
| 역별 프로필 | 0.150806837 → 0.150481986 | 0.165727453 → 0.165233807 | 0.162339372 → 0.161900556 |
| 동춘 휴무 | 0.150344958 → 0.150296328 | 0.165112264 → 0.165069381 | 0.161781990 → 0.161729477 |
| 음력 연휴 | 0.150296328 → 0.150296328 | 0.165069381 → 0.164734305 | 0.161729477 → 0.161376655 |
| 검단 아침 | 0.150296328 → 0.150203833 | 0.164734305 → 0.164714718 | 0.161376655 → 0.161375009 |
| TabICL 고정 혼합 검증 | 0.150481986 → 0.150344958 | 0.165233807 → 0.165112264 | 0.161900556 → 0.161781990 |
| TabICL 가중치 단독 비교 | 0.150344958 → 0.150177519 | 0.165112264 → 0.165071156 | 0.161781990 → 0.161676612 |
| 마지막 벡터 모형 | 0.150036236 → 0.149904003 | 0.164673510 → 0.164576151 | 0.161269365 → 0.161174942 |


TabICL 두 행은 역별 프로필 뒤의 고정 혼합과 가중치 후보를 따로 비교한 검증입니다. 휴무·연휴·아침 피크가 추가된 최종 결합의 점수와 직접 합산하지 않습니다. 마지막 벡터 행은 다른 후속층까지 적용한 상태에서 벡터를 추가한 비교입니다.


2026-03-31과 04-30 보조 기준일은 각각 4~6월, 5~6월을 예측하며 A와 겹칩니다. 원본 기본 신경망은 이 보조 구간에서 시드 0 하나를 사용했습니다. 역별 프로필 RMSLE는 3월 말 `0.149896238 → 0.149469121`, 4월 말 `0.143879163 → 0.143678155`입니다. 이어 TabICL 고정/기준일별 가중치를 적용한 3월 말은 `0.149469121 → 0.149121717`, 4월 말은 `0.143678155 → 0.143646169`입니다. 4월 말의 가중치 후보는 고정 혼합 `0.143625331`보다 나빴습니다. 최종 벡터까지의 해당 두 구간 수치는 기존 보고서에 없으므로 만들어 적지 않았습니다.

추가 행사 검증은 해당 행사의 과거 비교 날짜를 사용합니다. 예를 들어 송도맥주축제 2025년 행사 기간 두 역은 `0.52022 → 0.37353`, 부천 만화축제 2025-09-27~28 3759역은 `0.37523 → 0.29667`이었습니다. 송도 21~22시 선택은 같은 축제를 다시 본 결과이므로 독립 검증이 아닙니다. 검암·선학·둘레길의 표본과 결과는 `docs/sincon_clean_refined_results.json`의 `details`에 남겼습니다. 드문 행사에는 작은 검증 표본과 과적합 위험이 남습니다.

## full 단계별 스위치와 예상 시간

다음 시간은 실행 전 계획용 범위이며 측정값이 아닙니다. 실제 완료 시간은 `run.json` 및 `logs/`에 기록합니다. 사용한 머신에서 실행하지 않은 단계는 `STATUS.md`에 미실행으로 표시합니다. GPU는 CUDA만 사용하고 MPS에는 의존하지 않습니다. TabICL은 원래 재현 설정대로 CPU를 사용합니다.

| 단계 | 입력 → 출력 | 설정 | 계획용 예상 시간, 미측정 |
|---|---|---|---|
| native | 공식 train + 팀 외부 피처 → 모델별 최종·A/B/C 로그 예측 | LGB 시드0/1/2, 1500회. MLP 시드0~29, 30epoch, 폭512, dropout0.15 | GPU 20~120분, CPU 2~8시간 |
| stacks | 모델별 예측 + 비지하철 이동·버스·경기 → conditional/local 예측·A/B/C | 그래프120epoch 2시드, 혼합125epoch 2시드 | GPU 5~30분, CPU 10~90분 |
| rolling | 공식 과거 train → 6개 월말 예측, 엄격 여름 예측 | 월말 MLP 시드0·폭384. 엄격 여름 30시드·폭512 | GPU 20~120분, CPU 2~8시간 |
| hourly_xgb | 공식 과거 train → A/B/C·추가 기준일·최종 | 시드0, 2000회, 깊이8 | CPU 10~90분 |
| clean_oof | 새 과거 예측 + 공식 과거 정답 → 보정 전 OOF | 아홉 시간순 구간 | CPU 1~10분 |
| event_responses | 공식 train + 행사 플래그 변경 → 취소·장소별 모델 반응 | LGB 3시드, MLP 0/1/2, 원본 epoch·폭 | GPU 10~40분, CPU 30~180분 |
| tabicl | 공식 train + 외부 피처 + 동결 checkpoint → 10개 구간의 역·일 예측 | 시드2026, 앙상블2, CPU | CPU 5~60분 |
| vector | 공식 train + conditional 예측 → 18시간 차이 | 시드19, 650회, 깊이6 | CPU 5~60분 |
| 보정·검증 | 위 예측 → 최종 CSV·형식 검사·기준 차이 | 설정 및 검증에 따라 행사 보호 | CPU 수 분 목표 |

전체 CPU 실행은 수 시간 이상 걸릴 수 있습니다. 중간 결과는 새 실행 폴더에 보존되며, 다른 실행 폴더의 데이터와 덮어쓰지 않습니다. 단계를 끄면 원래 동결 캐시를 사용합니다. 다만 native를 켰다면 event_responses도 함께 켜야 합니다. 행사 응답은 원본 모델 예측과 일치해야 하므로 이 조합은 시작 전에 검사합니다. 이전 실행의 부분 결과로 자동 재개하는 기능은 제공하지 않습니다.


In [ ]:
# assemble에서는 학습하지 않습니다. full에서는 FULL_STAGES의 선택을 기록하고 순서대로 실행합니다.
run.train(FULL_STAGES)


# 외부 자료와 이용 조건

이 표는 2026-11-12에 팀이 작성·재확인한 출처 기록을 옮긴 것이다. 이번 패키지 작업에서는 새 외부 데이터를 수집하지 않았다. 최초 조회일이 없는 항목은 추측하지 않았다. 전체 개별 행사 URL과 원문 조건은 `docs/SINCON_EXTERNAL_SOURCES.md`에도 보존했다. 아래의 '7월 이후'는 2026년 7월 이후 정보다. 외부 날씨·버스·일정 등이 포함되지만, 지하철 승하차 실측은 포함하지 않는다.

| 이름·보관 위치·역할 | 출처 링크 | 조회일 | 이용 조건 | 7월 이후 정보 |
|---|---|---|---|---|
| 공휴일·선거일·연휴, `calendar_features.csv` | [법률](https://www.law.go.kr/법령/공휴일에관한법률), [월력요항](https://astro.kasi.re.kr/life/post/calendardata) | 최초 2026-11-11, 재확인 11-12 | 공개 날짜를 코드로 변환. 이미지·달력 표현물은 미사용 | 포함, 예정 날짜 |
| NEIS 학사일정, `school_calendar.csv` | [데이터포털](https://www.data.go.kr/data/15137088/openapi.do), [API](https://open.neis.go.kr/hub/SchoolSchedule) | 최초 2026-11-11, 재확인 11-12 | 포털 이용허락범위 제한 없음, API 호출 조건 적용 | 포함 |
| 대학 방학 대리변수 | [인천대 공문](https://liberaledu.inu.ac.kr/bbs/inu/246/379145/download), [법령](https://www.law.go.kr/법령/고등교육법시행령) | 최초 2026-11-11, 재확인 11-12 | 날짜 사실 사용. 대학 첨부문서의 별도 조건 미확인 | 포함 |
| Open-Meteo 시간별 날씨, 팀 1지점·후속 4권역 | [자료](https://open-meteo.com/en/docs/historical-weather-api), [이용 조건](https://open-meteo.com/en/features#terms) | 최초 2026-11-11, 재확인 11-12 | CC BY 4.0. Open-Meteo·Copernicus/ECMWF 출처 표기. 권역 매핑·집계 가공 | 포함, 실제 날씨 |
| 서울시·KT OA-22655 비지하철 이동량, `mobility_monthly/` | [자료·조건](https://data.seoul.go.kr/dataList/OA-22655/F/1/datasetView.do) | 최초 기록 없음, 재확인 2026-11-12 | 공공누리 제1유형. 서울특별시·KT 출처 표기. 광역버스4·일반버스5·도보7·차량8만 집계 | 포함, 실제 비지하철 이동량 |
| 인천교통공사 역 주소·출입구 좌표 | [역 주소](https://www.data.go.kr/data/15043811/fileData.do), [출입구](https://www.data.go.kr/data/15166820/fileData.do) | 최초 기록 없음, 재확인 2026-11-12 | 포털 이용허락범위 제한 없음. 주소·좌표만 사용 | 조회 시점의 정적 정보, 최초 갱신일 미확인 |
| 인천 버스 정류장별 월간 승하차·정류장 좌표 | [월간 버스](https://www.data.go.kr/data/15048264/fileData.do), [좌표](https://www.data.go.kr/data/15067528/fileData.do) | 최초 기록 없음, 재확인 2026-11-12 | 포털 이용허락범위 제한 없음 | 포함, 실제 버스 이용량 |
| K-apt 35개 단지 승인·입주월·세대수·좌표 | [실제 웹 출처](https://www.k-apt.go.kr/kaptinfo/openKaptLocation.do), [기본정보](https://www.data.go.kr/data/15073271/fileData.do) | 최초 기록 없음, 포털 조건 재확인 2026-11-12 | 포털 기본정보는 제한 없음. 실제 웹 지도 응답의 입주월·좌표에 같은 조건이 적용되는지 미확인 | 입주월에 이후 기간 포함. 지하철 관측 없음 |
| 문체부 지역축제 계획·축제 표준데이터·전년 관람객 대리값 | [문체부](https://www.mcst.go.kr/site/s_culture/festival/festivalList.jsp?pMenuCD=0304000000), [표준데이터](https://www.data.go.kr/data/15013104/standard.do) | 최초 기록 없음, 배포 페이지 재확인 2026-11-12 | 첨부파일별 공공누리 유형 미확인. 문체부·한국관광공사·제공 지자체 표기 | 포함, 일정·행사 사실 |
| KBO 문학 경기·실제 결과·관중·종료 시각 | [일정](https://www.koreabaseball.com/Schedule/Schedule.aspx), [관중](https://www.koreabaseball.com/Record/Crowd/GraphDaily.aspx), [약관](https://www.koreabaseball.com/Etc/Policy.aspx) | 팀 최초 2026-11-11, 별도 게임센터 최초 기록 없음, 조건 재확인 11-12 | 수집·이용 허용 미확인. 자동 수집·재배포 제한과 실제 경로의 관계 확인 필요 | 포함, 실제 경기 정보 |
| K리그 부천 경기·킥오프·관중 | [구단](https://www.kleague.com/club/club.do?teamId=K26), [약관](https://www.kleague.com/join/step1.do) | 최초 기록 없음, 재확인 2026-11-12 | 비상업 학습·재배포 허용 미확인 | 포함, 실제 경기 정보 |
| 팀 수기 공연·행사, `events.csv`, `events_daily.csv` | [주최 측 예시](https://www.songdobeer.com/), 행사별 전체 URL은 아래 부록 | 공통 최초 2026-11-11, 개별 조회일 기록 없음 | 일정·장소·시각 사실만 사용. 개별 페이지 이용 조건 미확인 | 포함, 일정·취소·개최 사실 |
| 취소·장소·접근·종료 등 후속 행사 공지 | [OBJET 주최](https://btobcompany.kr/notice/?bmode=view&category=&idx=174155451), [연수문화재단](https://www.ysfac.or.kr/user/cultural/events_view.php?sq=381), [과학대제전](https://www.icemf.kr/) | 최초 기록 없음, 팀 출처 정리 2026-11-12 | 공지의 사실만 입력. 개별 원문의 포괄적 이용 조건 미확인 | 포함, 실제 취소·위치·프로그램 등 |
| 동춘 대형마트 정기 휴무 | [연수구 운영표](https://www.yeonsu.go.kr/main/general/civil/manless.asp) | 재확인 2026-11-12 | 공식 표의 둘째·넷째 일요일 규칙을 사용. 페이지 개별 라이선스 미확인 | 정기 규칙의 예측 기간 적용 |
| TabICL 2.2.0 추론 코드 | [배포](https://pypi.org/project/tabicl/2.2.0/), [LICENSE](https://github.com/soda-inria/tabicl/blob/main/LICENSE) | 최초 기록 없음, 메타데이터 확인 2026-11-12 | 핵심 회귀 경로 BSD-3-Clause. 패키지에 포함한 라이선스 고지 보존 | 외부 사전학습 모델, 지하철 후속 실측 입력 없음 |
| TabICL 회귀 체크포인트 `20260212` | [공식 파일·해시](https://huggingface.co/jingang/TabICL/blob/main/tabicl-regressor-v2-20260212.ckpt) | 최초 기록 없음, 라이선스·해시 확인 2026-11-12 | 파일 페이지 BSD-3-Clause. SHA-256 `0db9cb538f114e79026bf08f45f41ad8dd7ad2de2aaca9a5ca8cd3bd9748ae7a` | 체크포인트 이름은 2026-02-12. 학습 원자료 전체에 대한 별도 감사는 미실시 |

공식 `train.csv`는 2026-06-30까지 688,176행이고, `test.csv`는 157,194개의 날짜·역·시간·ID만 제공한다. 공식 CSV는 대회 참가자가 운영진 계정의 비공개 Dataset에 올리는 용도다. 일반 공개 재배포 허락으로 간주하지 않는다.

이용 조건 미확인 자료를 전부 제외한 결과 차이는 미계산이다. 보관 예측 안에 이미 그 자료의 영향이 들어 있으므로, 입력 CSV만 삭제한 조립을 자료 제거 실험으로 표시할 수 없다. KBO·K리그·주거·행사별 원본 모델과 검증 캐시까지 다시 학습해야 한다. 이번 패키지는 기준 정리본을 보존하며, 해당 자료의 규칙 3절 적합성을 보증하지 않는다. 후속 행사층만 뺀 조건부 차이는 별도 검증 보고서로 계산하며, 외부 자료 전체 제거 효과와 구분한다.


## 실행해서 계산한 제한적 제거 비교

마지막 행사층만 끄고 다른 입력과 상위 모델을 유지하면 59행이 달라집니다.
최대 log1p 차이는 `0.274045497`, 전체 157,194행의 로그 차이 RMS는
`0.002804434`입니다. 최대 승차 인원 차이는
`355.567769`명입니다. 검암43·과학12·선학2·둘레길2행입니다.

이것은 해당 공지나 행사 자료 전체를 제거한 실험이 아닙니다. 상위 모델과 과거 검증 캐시에
이미 들어간 행사 정보는 유지했습니다. 미래 지하철 실측을 읽지 않았으므로 이 비교로 최종
RMSLE 개선량을 말할 수 없습니다. 이용 조건 미확인 자료 전체 제거의 영향은 미계산입니다.
근거 파일은 `verification/conditional_event_ablation.json`입니다.


## 시드·버전·하드웨어 차이

패키지 설치 버전은 첫 셀과 `requirements.txt`에 고정했습니다. Python 3.11의 numpy는 2.3.5, Python 3.12의 numpy는 2.5.4입니다. 최초 기준 출력의 일부 원본 모형은 numpy2.3.5·pandas2.2.3·torch2.9.1·LightGBM4.6.0에서 만들었습니다. 후속 보정의 기존 확인 환경은 Python3.12, numpy2.5.4·pandas3.1.1·scipy1.17.1입니다. 후기 XGBoost 캐시는3.4.1을 사용했으므로 Python3.12에는 같은3.4.1을 고정합니다. 이 버전은 Python3.12 이상을 요구합니다. Python3.11에서는 설치 가능한3.1.2를 사용합니다. 전체 재학습 재현에는 Python3.12를 권장합니다.

같은 입력·시드19·650회·깊이6으로 버전만 대조했습니다. XGBoost3.4.1의 최종 벡터 원시 예측은 보관 파일과 차이0이었고,3.1.2는 최대 차이0.791937197·평균 절대 차이0.013520701이었습니다. 원시 차이는 뒤에서 제한·평균 보존을 거쳐 적용됩니다. TabICL과 벡터만3.1.2 환경에서 재실행한 조립의 기준 대비 최대 log1p 차이는0.054610915, 전체 로그 차이 RMS는0.002170552였습니다. 이것은 Public 점수나 정답 오차가 아니라 두 예측 파일의 차이입니다. Python3.11 자체의 실제 실행은 미확인이며, 이 대조는 Python3.12에서 XGBoost 버전을 바꿔 측정했습니다. assemble은 원시 캐시를 사용하므로 이 트리 재학습 차이를 만들지 않습니다.

기본 Python3.12·XGBoost3.4.1에서 TabICL과 벡터를 모두 재실행한 뒤 조립한 결과는 기준 대비 최대 log1p 차이6.8664551e-7, 전체 로그 차이 RMS6.4427340e-8이었습니다. 원본30시드 MLP·스택 등 나머지 예측은 캐시를 사용한 부분 재학습입니다. 전체 실행 범위와 측정 시간은 `verification/MEASURED_RUNS.md` 및 `STATUS.md`에서 확인합니다.

신경망은 각 fit에서 `torch.manual_seed`를 호출합니다. GPU/CPU의 행렬 연산, PyTorch 버전, dropout 및 병렬 연산 순서가 달라지면 같은 시드라도 결과가 달라질 수 있습니다. 다른 장치에서의 전체 30시드 결과 차이는 아직 측정하지 않았습니다. '조금 달라진다'는 이유만으로 허용 크기를 임의로 정하지 않았습니다. 실제 실행한 모델별 최대·평균 차이만 검증 보고서에 씁니다. 캐시 assemble은 기준과 `max |log1p 차이| <= 1e-10`을 비교 기준으로 사용합니다.

이번 패키지의 Torch2.9.1 CPU에서 TabICL 최종 8,733역·일을 실제로 재실행했습니다. 기존 원시 예측과 최대 로그 차이는 `4.2915344e-6`, 평균 절대 차이는 `3.0683458e-7`이었습니다. 이 값은 현재 머신·패키지 조합에서의 측정값이며 다른 GPU나 30시드 MLP의 오차 한계가 아닙니다. 과거 구간을 포함한 단계 전체의 완료 여부는 `STATUS.md`와 `verification/full/`에 별도로 기록합니다.

행사 응답 MLP의 시드0·1·2도 원래 설정으로 CPU 재학습했습니다. 동일한 세 시드의 보관 응답과 비교하면, 문학 취소72행의3시드 평균 `combined_delta`는 최대0.330746810·평균 절대0.151959410 로그만큼 달랐습니다. 송도 장소144개 시나리오 행의3시드 평균 응답은 최대0.201554616·평균 절대0.054625309 로그만큼 달랐습니다. 실제 적용은 이 원시 응답을 다른 모델과 혼합하고 정해진 시간대에만 적용하므로 이 수치는 최종 CSV의 차이가 아닙니다. 장치와 라이브러리 버전 영향을 따로 분리한 실험도 아닙니다. 같은 시드라는 이유로 full 결과가 기준 CSV와 거의 같다고 단정할 수 없습니다. 기본 assemble은 이 재학습 응답을 사용하지 않고 검증된 보관 응답을 사용합니다.

Python3.11·Linux·Kaggle·Colab 실측 여부는 `STATUS.md`에 구분합니다. Linux 정적 점검은 대소문자·파일 해시·절대 경로·MPS·폰트 의존성을 확인하지만 클라우드 실행 성공을 대신하지 않습니다.

## Public 점수 사용 감사

팀의 옛 파일에는 Public 점수 차이로 역산한 전체 로그 수준과 역 패턴 배율이 있었습니다. 초기 앙상블의 `+0.0217424`, 후기 로컬 스택의 `+0.00902382`, 역 패턴의 `0.62`를 이번 경로에 사용하지 않습니다. 보관 ZIP도 필요한 외부 자료·모델별 예측만 남기고 옛 제출 CSV를 제거했습니다. 원본 팀 `ensemble.py`는 모델 특징·예측 함수만 전달하고 옛 제출 main은 제거했습니다. 호출되지 않아야 하는 옛 `calibrate`는 실행하면 오류를 내도록 막았습니다.

행사 취소·장소 조사의 우선순위를 정할 때 과거 Public 차이를 참고한 경위는 숨기지 않습니다. 현재 보정량은 공지와 동일 모델의 입력 변경 응답 및 6월30일 이전 공식 검증에서 재계산합니다. Public 점수는 완성한 후보를 고르는 데만 사용할 수 있습니다. 이 노트북에는 점수를 예측 함수로 전달하거나, 점수 변화로 정답·가중치를 역산하거나, Kaggle에 업로드하는 단계가 없습니다. 감사 원문은 `docs/SINCON_PUBLIC_SCORE_AUDIT.md`입니다.

## 마지막 보정 목록을 추가하는 방법

기본 행사·일반일층은 재현 순서대로 실행합니다. 이후 추가 보정만 `CORRECTIONS`로 제어합니다. CSV 형식은 `example_id,log_delta`이며, 검증된 로그 차이를 더합니다. 음수가 되면 로그값을 0으로 잘라 비음수 인원을 보장합니다. 활성화 시 CSV와 공식 과거 검증 근거 JSON을 둘 다 제공해야 합니다.

근거 JSON의 필수 내용은 `status: "validated"`, `label_source: "official_train"`, `uses_public_score: false`, `validated_through: "2026-06-30"` 이하 날짜, 그리고 `offset_sha256`입니다. 이 검사는 근거의 내용 자체를 심사하지 않습니다. 실제 효과 검토와 이용 조건 판단은 Claude가 해야 합니다. 현재 추가 보정 2개는 모두 꺼져 있습니다.


In [ ]:
final_csv = run.assemble(CORRECTIONS)


In [ ]:
verification = run.verify()
if RUN_MODE == "assemble" and not any(item["enabled"] for item in CORRECTIONS):
    assert verification["max_abs_log1p_difference"] <= 1e-10, "기준 파일과 재현 차이가 큽니다."
# 파일은 run.output 아래 있습니다. Kaggle은 오른쪽 Output에서 CSV를 내려받습니다.
from IPython.display import FileLink, display
display(FileLink(str(final_csv)))


## 결과를 검토할 때

`verification.json`에서 행 수·ID 순서·비음수 유한값·기준 최대 로그 차이를 확인하세요. `run.json`에는 실제 수행 단계와 경과 시간이 있고, 전체 학습이 아닌 경우 캐시 사용을 표시합니다. `early.json`, `events_before_model.json`, `model.json`, `events_after_model.json`에는 각 보정의 입력 해시와 변경 범위가 있습니다. `model_oof.csv.gz`는 검증 효과를 다시 계산할 때 사용할 수 있습니다.

재현 성공은 외부 자료의 이용 허락이 모두 확인되었다는 뜻이 아닙니다. KBO·K리그·일부 주거·행사 자료의 조건과 최종 후보 선택은 Claude의 검토가 남아 있습니다. 최신 실행 범위와 남은 위험은 동봉한 `STATUS.md`를 확인하세요.
